# Artificial Intelligence — Lab 3
## Uniform-Cost Search and Search Performance

**Course Learning Outcomes — CLO2 and CLO3**

- **CLO2:** Determine appropriate uninformed search strategies.
- **CLO3:** Illustrate and analyze the performance of search methods.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook containing predictions, traces, code, experiments, justifications, debugging answers, and reflection.

> **Assessment principle:** Correct code is only part of the evidence. Most marks come from your ability to **reason about path cost, predict frontier behavior, justify priority-queue decisions, interpret performance metrics, and explain why UCS differs from BFS**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Cost-based search recap | 10 min | Distinguish depth from cumulative path cost |
| 2. Manual UCS trace | 20 min | Predict priority-queue evolution before coding |
| 3. UCS implementation | 30 min | Implement graph search with `heapq` |
| 4. BFS vs. UCS experiment | 25 min | Compare path length, path cost, and search effort |
| 5. Performance instrumentation | 20 min | Measure expanded nodes and frontier size |
| 6. Debugging, variation & reflection | 15 min | Diagnose common errors and defend conclusions |

> **Main idea:** Uniform-Cost Search expands the frontier node with the smallest cumulative path cost $g(n)$, not the shallowest node.

## Learning Objectives

By the end of this lab, you should be able to:

1. compute cumulative path cost $g(n)$;
2. explain why a FIFO queue is insufficient for weighted search;
3. manually trace Uniform-Cost Search on a weighted graph;
4. implement UCS using a priority queue;
5. explain the role of the best-known cost table;
6. compare BFS and UCS on the same weighted problem;
7. distinguish **solution depth** from **solution cost**;
8. measure basic search-performance indicators;
9. diagnose common UCS implementation errors;
10. justify when UCS is preferable to BFS.

In [ ]:
import heapq
from collections import deque
from itertools import count
from typing import Dict, List, Tuple, Optional

print("Lab 3 environment ready.")

# Part I — From Depth to Path Cost

In Lab 2, BFS selected nodes according to depth. That works well when every action has the same cost.

For weighted problems, we instead care about

$$
g(n)=\text{cost from the initial state to node }n.
$$

Uniform-Cost Search selects the frontier node with the smallest $g(n)$.

Consider this weighted graph:

```text
        1          5
    S ------ A -------- G
     \        \
      \4       \2
       \        \
        B --1--- D --1-- G
```

We will use the following directed edges:

- $S \rightarrow A$ with cost 1
- $S \rightarrow B$ with cost 4
- $A \rightarrow G$ with cost 5
- $A \rightarrow D$ with cost 2
- $B \rightarrow D$ with cost 1
- $D \rightarrow G$ with cost 1

In [ ]:
WEIGHTED_GRAPH = {
    "S": [("A", 1), ("B", 4)],
    "A": [("G", 5), ("D", 2)],
    "B": [("D", 1)],
    "D": [("G", 1)],
    "G": [],
}

START = "S"
GOAL = "G"

## Task 1.1 — Compare Candidate Paths

Before running any search algorithm, compute the path cost of each route.

| Path | Number of actions | Total cost |
|---|---:|---:|
| $S \rightarrow A \rightarrow G$ |  |  |
| $S \rightarrow A \rightarrow D \rightarrow G$ |  |  |
| $S \rightarrow B \rightarrow D \rightarrow G$ |  |  |

Then answer:

1. Which path has the **fewest actions**?
2. Which path has the **lowest total cost**?
3. Why can these be different?
4. Which quantity does BFS primarily optimize in an unweighted graph?
5. Which quantity does UCS optimize under nonnegative step costs?

**Your answers:**

# Part II — Manual Uniform-Cost Search Trace

For the trace below, assume:

- the frontier is ordered by cumulative path cost $g$;
- the start state has $g(S)=0$;
- when a cheaper path to a frontier state is found, its best-known cost is updated;
- the goal test is applied when the node is removed from the frontier for expansion.

## Task 2.1 — Predict the UCS Trace

Complete the table **before implementing UCS**.

| Step | Expanded node | $g$ of expanded node | Frontier after expansion, ordered by cost |
|---:|---|---:|---|
| 0 | — | — | `[(S,0)]` |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |
| 4 |  |  |  |

Then answer:

1. Which path do you predict UCS will return?
2. What will its total cost be?
3. At what cost is `D` first discovered?
4. Is a cheaper path to `D` later found? Explain.

**Your answers:**

## Task 2.2 — Why a Priority Queue?

Explain in 2–4 sentences:

1. Why is `deque.popleft()` not sufficient for UCS?
2. What key should determine the priority of a frontier entry?
3. Why is the cumulative cost $g(n)$ different from the immediate edge cost?

**Your answer:**

# Part III — Implement Uniform-Cost Search

We will use Python's `heapq`.

A heap entry will have the form:

```python
(cost, tie_breaker, state)
```

The tie breaker prevents Python from needing to compare complex state objects when costs are equal.

The algorithm must return:

```python
(path, total_cost, stats)
```

where `stats` includes basic search-performance information.

In [ ]:
def reconstruct_path(parent: Dict[str, Optional[str]], goal: str) -> List[str]:
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path

## Task 3.1 — Complete UCS

### Required behavior

Your implementation should:

- insert the start with cost 0;
- always pop the smallest-cost frontier entry;
- maintain `best_cost[state]`;
- update a parent when a cheaper path is found;
- ignore stale heap entries whose cost is no longer the best known;
- stop when the goal is popped;
- count expanded nodes;
- track the maximum frontier size.

In [ ]:
def uniform_cost_search(graph, start, goal):
    tie = count()

    frontier = []
    heapq.heappush(frontier, (0, next(tie), start))

    best_cost = {start: 0}
    parent = {start: None}

    expanded_nodes = 0
    max_frontier_size = 1
    expansion_order = []

    while frontier:
        # TODO 1: pop the lowest-cost entry
        cost, _, current = None, None, None

        # TODO 2:
        # If this is a stale entry, skip it.
        # Hint: compare 'cost' with best_cost[current].

        # TODO 3:
        # Record the expansion.
        # expanded_nodes += 1
        # expansion_order.append((current, cost))

        # TODO 4:
        # If current == goal, reconstruct and return:
        # path, cost, stats

        # TODO 5:
        # For each (child, step_cost) in graph[current]:
        #   new_cost = cost + step_cost
        #   if child has no known cost OR new_cost is cheaper:
        #       update best_cost[child]
        #       update parent[child]
        #       push the new entry into the heap

        # TODO 6:
        # update max_frontier_size
        pass

    return None, float("inf"), {
        "expanded_nodes": expanded_nodes,
        "max_frontier_size": max_frontier_size,
        "expansion_order": expansion_order,
    }

### Task 3.2 — Predict Before Running the Self-Check

Write your prediction first:

- **Expected path:**  
- **Expected cost:**  
- **Expected first three expansions with costs:**  

Then run the next cell.

In [ ]:
ucs_path, ucs_cost, ucs_stats = uniform_cost_search(
    WEIGHTED_GRAPH, START, GOAL
)

print("UCS path:", ucs_path)
print("UCS cost:", ucs_cost)
print("Expansion order:", ucs_stats["expansion_order"])
print("Expanded nodes:", ucs_stats["expanded_nodes"])
print("Maximum frontier size:", ucs_stats["max_frontier_size"])

assert ucs_path == ["S", "A", "D", "G"]
assert ucs_cost == 4
assert ucs_stats["expansion_order"][0] == ("S", 0)

print("UCS tests passed.")

## Task 3.3 — Explain the Implementation

Answer in your own words.

1. What does `best_cost[state]` represent?
2. Why do we update a state's parent only when a cheaper path is found?
3. What is a **stale heap entry**?
4. Why can stale entries appear even when `best_cost` is correct?
5. Why is the goal test safest when the goal is popped from the priority queue rather than immediately when first generated?

**Your answers:**

# Part IV — BFS vs. UCS on a Weighted Graph

To compare the strategies fairly, we also implement a simple BFS that ignores edge weights while searching.

It still computes the final cost of the path it returns.

In [ ]:
def path_cost(graph, path):
    if path is None:
        return float("inf")

    total = 0

    for current, nxt in zip(path, path[1:]):
        found = False
        for child, edge_cost in graph[current]:
            if child == nxt:
                total += edge_cost
                found = True
                break

        if not found:
            raise ValueError(f"No edge {current!r} -> {nxt!r}")

    return total


def bfs_weighted_graph(graph, start, goal):
    frontier = deque([start])
    discovered = {start}
    parent = {start: None}

    expanded_nodes = 0
    max_frontier_size = 1
    expansion_order = []

    while frontier:
        current = frontier.popleft()
        expanded_nodes += 1
        expansion_order.append(current)

        if current == goal:
            path = reconstruct_path(parent, goal)
            return path, path_cost(graph, path), {
                "expanded_nodes": expanded_nodes,
                "max_frontier_size": max_frontier_size,
                "expansion_order": expansion_order,
            }

        for child, _ in graph[current]:
            if child not in discovered:
                discovered.add(child)
                parent[child] = current
                frontier.append(child)

        max_frontier_size = max(max_frontier_size, len(frontier))

    return None, float("inf"), {
        "expanded_nodes": expanded_nodes,
        "max_frontier_size": max_frontier_size,
        "expansion_order": expansion_order,
    }

## Task 4.1 — Predict BFS vs. UCS

Before executing the comparison, complete:

- **I predict BFS will return:**  
- **Predicted BFS path cost:**  
- **I predict UCS will return:**  
- **Predicted UCS path cost:**  
- **Which algorithm should return the cheaper solution? Why?**  

Then run the experiment.

In [ ]:
bfs_path, bfs_cost, bfs_stats = bfs_weighted_graph(
    WEIGHTED_GRAPH, START, GOAL
)

ucs_path, ucs_cost, ucs_stats = uniform_cost_search(
    WEIGHTED_GRAPH, START, GOAL
)

print("BFS")
print("  path:", bfs_path)
print("  actions:", len(bfs_path) - 1)
print("  cost:", bfs_cost)
print("  expanded:", bfs_stats["expanded_nodes"])
print("  max frontier:", bfs_stats["max_frontier_size"])

print("\nUCS")
print("  path:", ucs_path)
print("  actions:", len(ucs_path) - 1)
print("  cost:", ucs_cost)
print("  expanded:", ucs_stats["expanded_nodes"])
print("  max frontier:", ucs_stats["max_frontier_size"])

## Task 4.2 — Interpret the Comparison

Complete the table.

| Metric | BFS | UCS |
|---|---:|---:|
| Number of actions |  |  |
| Path cost |  |  |
| Expanded nodes |  |  |
| Maximum frontier size |  |  |

Then answer:

1. Did BFS return the path with the fewest actions?
2. Did BFS return the cheapest path?
3. Did UCS return the cheapest path?
4. Why does this experiment demonstrate that **depth and cost are different criteria**?
5. Can you conclude from one small graph that UCS always expands fewer nodes than BFS? Why not?

**Your answers:**

# Part V — Search Performance on a Weighted Grid

We now move to a grid with nonuniform terrain costs.

Legend:

- `S` = start
- `G` = goal
- `#` = obstacle
- `.` = normal cell, entry cost 1
- `r` = rough terrain, entry cost 4
- `m` = mud, entry cost 7

A search may choose a longer route in number of actions if that route is cheaper.

In [ ]:
GRID = [
    ".......G",
    ".###....",
    ".r.r....",
    ".r.#.m..",
    ".r.#.m..",
    "S..#....",
]

GRID_START = (5, 0)
GRID_GOAL = (0, 7)

MOVES = [
    ("Up", (-1, 0)),
    ("Right", (0, 1)),
    ("Down", (1, 0)),
    ("Left", (0, -1)),
]


def terrain_cost(cell):
    if cell == "r":
        return 4
    if cell == "m":
        return 7
    return 1


def grid_neighbors(state):
    r, c = state
    result = []

    for action, (dr, dc) in MOVES:
        nr, nc = r + dr, c + dc

        if (
            0 <= nr < len(GRID)
            and 0 <= nc < len(GRID[0])
            and GRID[nr][nc] != "#"
        ):
            result.append(((nr, nc), terrain_cost(GRID[nr][nc])))

    return result


def show_grid(path=None):
    canvas = [list(row) for row in GRID]

    if path:
        for r, c in path[1:-1]:
            if canvas[r][c] not in {"#", "S", "G"}:
                canvas[r][c] = "*"

    print("\n".join(" ".join(row) for row in canvas))


show_grid()

## Task 5.1 — Generalize UCS to the Grid

Complete the function below. It is the same UCS idea, but neighbors are now generated dynamically.

In [ ]:
def uniform_cost_grid(start, goal):
    tie = count()
    frontier = [(0, next(tie), start)]

    best_cost = {start: 0}
    parent = {start: None}

    expanded_nodes = 0
    max_frontier_size = 1
    expansion_order = []

    while frontier:
        # TODO: implement the UCS loop using grid_neighbors(current)
        pass

    return None, float("inf"), {
        "expanded_nodes": expanded_nodes,
        "max_frontier_size": max_frontier_size,
        "expansion_order": expansion_order,
    }

## Task 5.2 — Predict Before Executing the Weighted Grid

Before running UCS:

1. Do you expect the cheapest path to cross many `m` cells? Why or why not?
2. Can the cheapest path contain more actions than another available path?
3. Which metric should determine the better solution here: number of actions or total cost?

**Your prediction:**

In [ ]:
grid_path, grid_cost, grid_stats = uniform_cost_grid(
    GRID_START, GRID_GOAL
)

print("UCS weighted-grid path:", grid_path)
print("Number of actions:", None if grid_path is None else len(grid_path) - 1)
print("Total cost:", grid_cost)
print("Expanded nodes:", grid_stats["expanded_nodes"])
print("Maximum frontier size:", grid_stats["max_frontier_size"])

print("\nPath:")
show_grid(grid_path)

## Task 5.3 — Analyze Search Performance

Record:

| Measure | Result |
|---|---:|
| Solution actions |  |
| Solution cost |  |
| Expanded nodes |  |
| Maximum frontier size |  |

Then explain:

1. Which measure describes **solution quality** for this weighted grid?
2. Which measures describe **search effort**?
3. Why is maximum frontier size relevant to memory usage?
4. Why should execution time be interpreted cautiously when running small experiments on different computers?

**Your answers:**

# Part VI — Debugging UCS

## Task 6.1 — Faulty UCS: Immediate Edge Cost

A student writes:

```python
for child, step_cost in graph[current]:
    heapq.heappush(frontier, (step_cost, child))
```

Explain:

1. What cost is being used as the priority?
2. What cost **should** be used?
3. Give a short example of how this error could select the wrong node.

**Your answer:**

## Task 6.2 — Faulty UCS: First Path Wins

A student writes:

```python
if child not in discovered:
    discovered.add(child)
    best_cost[child] = new_cost
```

and never allows a cheaper later path to the same frontier state.

1. Why can this be incorrect for UCS?
2. Use node `D` in the lab graph to explain the issue.
3. What comparison should be performed before updating `best_cost[child]`?

**Your answer:**

## Task 6.3 — Goal Test Too Early

A student terminates UCS as soon as `G` is first generated.

1. Why can this be unsafe?
2. What must be true when the goal is **popped** from the frontier?
3. Under the usual UCS assumptions, why does that justify termination?

**Your answer:**

# Part VII — Personalized Cost Variation

Use the last digit of your student ID.

- `0–3`: change cost of $S \rightarrow B$ to **2**
- `4–6`: change cost of $A \rightarrow G$ to **2**
- `7–9`: change cost of $D \rightarrow G$ to **4**

Create a copy of the graph and modify only your assigned edge.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

personal_graph = {
    node: list(edges)
    for node, edges in WEIGHTED_GRAPH.items()
}

# TODO:
# Apply your assigned edge-cost modification to personal_graph.
# Keep WEIGHTED_GRAPH unchanged.

## Task 7.1 — Predict the Effect Before Running

Write:

- **My modified edge:**  
- **Original cost:**  
- **New cost:**  
- **Predicted optimal path:**  
- **Predicted optimal cost:**  
- **Reason:**  

Only after making the prediction should you run UCS.

In [ ]:
if LAST_DIGIT is not None:
    p_path, p_cost, p_stats = uniform_cost_search(
        personal_graph, START, GOAL
    )

    print("Personalized UCS path:", p_path)
    print("Personalized UCS cost:", p_cost)
    print("Expanded nodes:", p_stats["expanded_nodes"])
    print("Maximum frontier size:", p_stats["max_frontier_size"])
    print("Expansion order:", p_stats["expansion_order"])

## Task 7.2 — Explain the Personalized Result

1. Was your predicted path correct?
2. Was your predicted cost correct?
3. Did changing one edge cost change the **state space**, the **step-cost function**, or both?
4. Why can a small cost change alter the order in which UCS expands nodes?
5. If your optimal path did not change, does that mean the modification had no effect on search behavior? Explain.

**Your answers:**

# Part VIII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me the line that makes UCS choose the lowest-cost frontier node.
- What does `best_cost[state]` mean?
- Why do stale heap entries occur?
- Why can BFS return a more expensive solution?
- Why should UCS test the goal when it is popped?
- Which metric here measures memory pressure?
- If I change one edge cost, which part of $P=(S,A,s_0,T,G,c)$ changes?
- Explain the difference between step cost and cumulative path cost.

> You are expected to explain the **search concept represented by the code**, not memorize syntax.

# Reflection

Answer concisely but precisely.

### R1 — UCS vs. BFS
Under what condition will BFS and UCS behave equivalently with respect to solution quality?

**Answer:**

### R2 — Optimality
Why is UCS optimal when step costs are positive (or bounded below by a positive value)?

**Answer:**

### R3 — Priority
Why must the frontier be ordered by cumulative path cost rather than by the cost of the last action?

**Answer:**

### R4 — Performance
Why is “expanded fewer nodes” not the same as “returned a better solution”?

**Answer:**

### R5 — Strategy Selection
Give one example of a real problem where action costs are naturally unequal and UCS is more appropriate than BFS.

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] candidate-path cost calculations;
- [ ] manual UCS trace;
- [ ] justification for using a priority queue;
- [ ] working UCS implementation;
- [ ] explanation of `best_cost`, stale entries, and goal timing;
- [ ] BFS vs. UCS prediction and comparison;
- [ ] completed weighted-grid UCS;
- [ ] search-performance analysis;
- [ ] debugging answers for all three UCS errors;
- [ ] personalized cost variation;
- [ ] prediction made before the personalized run;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab03_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | UCS works correctly on graph and weighted grid |
| **Algorithmic justification** | **3.0** | Explains priority queue, cumulative cost, best-known costs, stale entries, and goal timing |
| **Experimental analysis** | **2.0** | Interprets BFS/UCS and weighted-grid performance results |
| **Trace / prediction / debugging** | **1.0** | Manual trace, predictions, and diagnosis of faulty UCS logic |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** Correct code without an adequate explanation earns only a limited portion of the marks.

## Key Takeaways

- BFS expands by **depth**; UCS expands by **cumulative path cost**.
- The priority used by UCS is

$$
g(n)=\sum c(s,a,s').
$$

- A priority queue is essential when frontier nodes have different costs.
- The best-known cost table allows UCS to improve a previously discovered path.
- A stale heap entry is an older, more expensive entry that should be ignored.
- On weighted problems, the path with the fewest actions may not be the cheapest path.
- Search evaluation should distinguish:
  - **solution quality**: path cost;
  - **search effort**: expanded nodes;
  - **memory pressure**: frontier size.

The next lab will introduce **Greedy Best-First Search and A\*** and study the effect of heuristic information.